# 00 — Exploração de domínio e climatologia (Fase 0)

Notebook mínimo para validar **configuração de paths** e, quando os dados
estiverem acessíveis na rede CENSIPAM, abrir uma amostra MERGE/ERA5.

**Expectativa fora da rede institucional:** as células de leitura falham com
mensagem clara — isso é normal. Não inventa arquivos nem resultados.

Pré-requisitos na máquina CENSIPAM:
1. `CENSIPAM_ERA5_ROOT` (e `CENSIPAM_MERGE_ROOT` quando conhecido)
2. Máscara Amazônia Legal + `CENSIPAM_AMAZONIA_LEGAL_MASK`
3. Checklist de padrões de nome preenchido em `docs/inventario-dados.md`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Raiz do repositório (sobe a partir de notebooks/)
REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from leka.config import load_paths, path_exists

paths = load_paths()
print("Paths resolvidos:")
for k, v in paths.summary().items():
    print(f"  {k}: {v}")
print()
print("ERA5 acessível?", path_exists(paths.era5_root))
print("MERGE acessível?", path_exists(paths.merge_root))
print("Máscara acessível?", path_exists(paths.amazonia_legal_mask))

## Máscara Amazônia Legal

Sem arquivo configurado, a célula abaixo apenas registra o aviso —
**não** gera geometria fictícia.

In [ ]:
from leka.domain.amazonia_legal import MaskUnavailableError, load_amazonia_legal_mask

try:
    gdf = load_amazonia_legal_mask(paths)
    print(f"Máscara carregada: {len(gdf)} feição(ões), CRS={gdf.crs}")
    display(gdf.head())
except MaskUnavailableError as exc:
    print(f"[esperado na Fase 0 sem máscara] {exc}")

## Amostra ERA5 / MERGE

Preencha `ERA5_RELATIVE` ou `ERA5_GLOBS` / equivalentes MERGE **somente**
depois de anotar padrões reais no inventário. Valores `None` = pulo gracioso.

In [ ]:
from leka.io.era5 import DataUnavailableError as Era5Unavailable
from leka.io.era5 import open_era5_sample
from leka.io.merge import DataUnavailableError as MergeUnavailable
from leka.io.merge import open_merge_sample

# Preencher na rede CENSIPAM após o inventário (não inventar nomes aqui):
ERA5_RELATIVE = None  # ex.: "subdir/arquivo.nc"
ERA5_GLOBS = None     # ex.: ("**/*sst*.nc",)
MERGE_RELATIVE = None
MERGE_GLOBS = None

def try_open(label, opener, **kwargs):
    try:
        ds = opener(paths, **kwargs)
        print(f"{label}: aberto — dims={dict(ds.dims)}, vars={list(ds.data_vars)[:8]}")
        ds.close()
    except (Era5Unavailable, MergeUnavailable, FileNotFoundError, OSError) as exc:
        print(f"{label}: indisponível nesta máquina — {exc}")
    except Exception as exc:  # noqa: BLE001 — notebook de diagnóstico
        print(f"{label}: falha ao abrir ({type(exc).__name__}): {exc}")

if ERA5_RELATIVE or ERA5_GLOBS:
    try_open("ERA5", open_era5_sample, relative_path=ERA5_RELATIVE, patterns=ERA5_GLOBS)
else:
    print("ERA5: pulado — defina ERA5_RELATIVE ou ERA5_GLOBS após o inventário.")

if MERGE_RELATIVE or MERGE_GLOBS:
    try_open("MERGE", open_merge_sample, relative_path=MERGE_RELATIVE, patterns=MERGE_GLOBS)
else:
    print("MERGE: pulado — defina MERGE_RELATIVE ou MERGE_GLOBS (e CENSIPAM_MERGE_ROOT).")

## Climatologia de precipitação (2001–2020)

Cálculo e mapas entram na **Fase 1**. Aqui só registramos o período de referência
configurado — sem inventar figuras.

In [ ]:
print(
    f"Climatologia de precip MERGE de referência: "
    f"{paths.precip_clim_start}–{paths.precip_clim_end}"
)
print(
    "Próximo passo (Fase 1, na rede CENSIPAM): ler MERGE no domínio Amazônia Legal, "
    "calcular climatologia sazonal e anomalias.")
